# IncidentIQ — Retrieval Evaluation

## Objective

Evaluate and compare the three retrieval approaches used in IncidentIQ:

1. BM25 lexical retrieval
2. Semantic retrieval
3. Hybrid retrieval using Reciprocal Rank Fusion (RRF)

The goal is to determine how each retrieval strategy performs
on representative incident-related queries.

In [2]:
from incidentiq.search import SearchEngine

from incidentiq.evaluation.metrics import (
    precision_at_k,
    recall_at_k,
    ndcg_at_k
)

In [5]:
DATA_PATH = "../data/processed/logs.parquet"

engine = SearchEngine(DATA_PATH)

print("Corpus size: ", len(engine.df))

Batches: 100%|██████████| 63/63 [00:04<00:00, 13.66it/s]

Corpus size:  2000


In [6]:
TEST_QUERIES = [
    "machine check timeout",
    "node card failure",
    "cache parity error",
    "network packet error",
    "floating point exception"
]

print("Evaluation queries:")

for i, query in enumerate(
    TEST_QUERIES,
    start=1
):
    print(f"{i}. {query}")

Evaluation queries:
1. machine check timeout
2. node card failure
3. cache parity error
4. network packet error
5. floating point exception


In [7]:
def retrieve_all(
    engine,
    query,
    top_k=10
):
    """
    Run BM25, Semantic, and Hybrid retrieval
    for the same query.
    """

    return {
        "BM25": engine.search_bm25(
            query,
            top_k=top_k
        ),

        "Semantic": engine.search_semantic(
            query,
            top_k=top_k
        ),

        "RRF": engine.search_hybrid(
            query,
            top_k=top_k
        )
    }

In [8]:
def display_results(
    query,
    results
):
    print("=" * 80)
    print(f"QUERY: {query}")
    print("=" * 80)

    for method, method_results in results.items():

        print(f"\n--- {method} ---")

        for result in method_results:

            print(
                f"{result['rank']:2}. "
                f"doc={result['doc_id']} | "
                f"{result['message']}"
            )

In [9]:
for query in TEST_QUERIES:

    results = retrieve_all(
        engine,
        query,
        top_k=10
    )

    display_results(
        query,
        results
    )

QUERY: machine check timeout

--- BM25 ---
 1. doc=260 | machine check enable..............0
 2. doc=261 | machine check enable..............0
 3. doc=1769 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 4. doc=1770 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 5. doc=1771 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 6. doc=1772 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 7. doc=1773 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 8. doc=1774 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 9. doc=1775 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
10. doc=206 | machine check: i-fetch......................0

--- Semantic ---
 1. doc=1774 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 2. doc=1770 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 3. doc=1775 |

In [10]:
qrels = {
    "machine check timeout": {
        # doc_id: relevance
    },

    "node card failure": {
        # doc_id: relevance
    },

    "cache parity error": {
        # doc_id: relevance
    },

    "network packet error": {
        # doc_id: relevance
    },

    "floating point exception": {
        # doc_id: relevance
    }
}

In [11]:
def inspect_results(results):
    
    for method, method_results in results.items():

        print(f"\n--- {method} ---")

        for result in method_results:

            print(
                f"{result['rank']:2}. "
                f"doc={result['doc_id']} | "
                f"{result['message']}"
            )

In [12]:
query = "machine check timeout"

results = retrieve_all(
    engine,
    query,
    top_k=10
)

inspect_results(results)


--- BM25 ---
 1. doc=260 | machine check enable..............0
 2. doc=261 | machine check enable..............0
 3. doc=1769 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 4. doc=1770 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 5. doc=1771 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 6. doc=1772 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 7. doc=1773 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 8. doc=1774 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 9. doc=1775 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
10. doc=206 | machine check: i-fetch......................0

--- Semantic ---
 1. doc=1774 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 2. doc=1770 | MACHINE CHECK DCR read timeout (mc=e08x iar 0x00000000 lr 0xc00045a4)
 3. doc=1775 | MACHINE CHECK DCR read timeo

In [13]:
def build_qrels(df):

    qrels = {}

    # =========================================================
    # 1. MACHINE CHECK TIMEOUT
    # =========================================================

    judgments = {}

    for doc_id, row in df.iterrows():

        message = row["message"].lower()

        if "machine check dcr read timeout" in message:
            judgments[doc_id] = 2

        elif "machine check" in message:
            judgments[doc_id] = 1

    qrels["machine check timeout"] = judgments


    # =========================================================
    # 2. NODE CARD FAILURE
    # =========================================================

    judgments = {}

    for doc_id, row in df.iterrows():

        message = row["message"].lower()

        if "node card is not fully functional" in message:
            judgments[doc_id] = 2

        elif "can not get assembly information for node card" in message:
            judgments[doc_id] = 2

        elif "node card vpd check" in message:
            judgments[doc_id] = 2

        elif "node card status" in message:
            judgments[doc_id] = 1

    qrels["node card failure"] = judgments


    # =========================================================
    # 3. CACHE PARITY ERROR
    # =========================================================

    judgments = {}

    for doc_id, row in df.iterrows():

        message = row["message"].lower()

        if (
            "cache" in message
            and "parity error" in message
        ):
            judgments[doc_id] = 2

        elif "cache" in message:
            judgments[doc_id] = 1

    qrels["cache parity error"] = judgments


    # =========================================================
    # 4. NETWORK PACKET ERROR
    # =========================================================

    judgments = {}

    for doc_id, row in df.iterrows():

        message = row["message"].lower()

        if "error receiving packet on tree network" in message:
            judgments[doc_id] = 2

    qrels["network packet error"] = judgments


    # =========================================================
    # 5. FLOATING POINT EXCEPTION
    # =========================================================

    judgments = {}

    for doc_id, row in df.iterrows():

        message = row["message"].lower()

        if (
            "floating point" in message
            and "exception" in message
        ):
            judgments[doc_id] = 2

        elif "floating point" in message:
            judgments[doc_id] = 1

    qrels["floating point exception"] = judgments


    return qrels

In [14]:
qrels = build_qrels(
    engine.df
)

In [15]:
from collections import Counter

for query, judgments in qrels.items():

    print(
        f"{query:30} | "
        f"total={len(judgments):3} | "
        f"{Counter(judgments.values())}"
    )

machine check timeout          | total= 14 | Counter({1: 7, 2: 7})
node card failure              | total= 27 | Counter({2: 21, 1: 6})
cache parity error             | total= 54 | Counter({2: 48, 1: 6})
network packet error           | total=  5 | Counter({2: 5})
floating point exception       | total=125 | Counter({2: 121, 1: 4})


In [16]:
K = 10

evaluation_results = {}

for query in TEST_QUERIES:

    results = retrieve_all(
        engine,
        query,
        top_k=K
    )

    judgments = qrels[query]

    evaluation_results[query] = {}

    for method, method_results in results.items():

        evaluation_results[query][method] = {
            "precision": precision_at_k(
                method_results,
                judgments,
                K
            ),

            "recall": recall_at_k(
                method_results,
                judgments,
                K
            ),

            "ndcg": ndcg_at_k(
                method_results,
                judgments,
                K
            )
        }

In [17]:
for query, methods in evaluation_results.items():

    print("=" * 70)
    print(f"QUERY: {query}")
    print("=" * 70)

    for method, metrics in methods.items():

        print(
            f"{method:8} | "
            f"P@10={metrics['precision']:.4f} | "
            f"Recall@10={metrics['recall']:.4f} | "
            f"nDCG@10={metrics['ndcg']:.4f}"
        )

QUERY: machine check timeout
BM25     | P@10=1.0000 | Recall@10=0.7143 | nDCG@10=0.8283
Semantic | P@10=1.0000 | Recall@10=0.7143 | nDCG@10=1.0000
RRF      | P@10=1.0000 | Recall@10=0.7143 | nDCG@10=0.9660
QUERY: node card failure
BM25     | P@10=1.0000 | Recall@10=0.3704 | nDCG@10=1.0000
Semantic | P@10=0.9000 | Recall@10=0.3333 | nDCG@10=0.7938
RRF      | P@10=1.0000 | Recall@10=0.3704 | nDCG@10=1.0000
QUERY: cache parity error
BM25     | P@10=1.0000 | Recall@10=0.1852 | nDCG@10=1.0000
Semantic | P@10=1.0000 | Recall@10=0.1852 | nDCG@10=1.0000
RRF      | P@10=1.0000 | Recall@10=0.1852 | nDCG@10=1.0000
QUERY: network packet error
BM25     | P@10=0.5000 | Recall@10=1.0000 | nDCG@10=1.0000
Semantic | P@10=0.5000 | Recall@10=1.0000 | nDCG@10=1.0000
RRF      | P@10=0.5000 | Recall@10=1.0000 | nDCG@10=1.0000
QUERY: floating point exception
BM25     | P@10=0.4000 | Recall@10=0.0320 | nDCG@10=0.0909
Semantic | P@10=1.0000 | Recall@10=0.0800 | nDCG@10=1.0000
RRF      | P@10=1.0000 | Recall@10

In [19]:
import pandas as pd
rows = []

for query, methods in evaluation_results.items():

    for method, metrics in methods.items():

        rows.append({
            "query": query,
            "method": method,
            "precision@10": metrics["precision"],
            "recall@10": metrics["recall"],
            "nDCG@10": metrics["ndcg"]
        })

evaluation_df = pd.DataFrame(rows)

evaluation_df

,query,method,precision@10,recall@10,nDCG@10
0,machine check timeout,BM25,1.0,0.714286,0.828346
1,machine check timeout,Semantic,1.0,0.714286,1.000000
2,machine check timeout,RRF,1.0,0.714286,0.965983
3,node card failure,BM25,1.0,0.370370,1.000000
4,node card failure,Semantic,0.9,0.333333,0.793765
5,node card failure,RRF,1.0,0.370370,1.000000
6,cache parity error,BM25,1.0,0.185185,1.000000
7,cache parity error,Semantic,1.0,0.185185,1.000000
8,cache parity error,RRF,1.0,0.185185,1.000000
9,network packet error,BM25,0.5,1.000000,1.000000


In [20]:
summary = (
    evaluation_df
    .groupby("method")[
        [
            "precision@10",
            "recall@10",
            "nDCG@10"
        ]
    ]
    .mean()
    .sort_values(
        "nDCG@10",
        ascending=False
    )
)

summary

,precision@10,recall@10,nDCG@10
method,,,
RRF,0.90,0.469968,0.993197
Semantic,0.88,0.462561,0.958753
BM25,0.78,0.460368,0.783847


## Results and Conclusion

We evaluated BM25, Semantic Retrieval, and Hybrid Retrieval using
Precision@10, Recall@10, and nDCG@10 across five representative
incident-related queries.

| Method | Precision@10 | Recall@10 | nDCG@10 |
|--------|--------------|-----------|---------|
| BM25 | 0.780 | 0.460 | 0.783 |
| Semantic | 0.880 | 0.463 | 0.959 |
| RRF | 0.900 | 0.470 | 0.993 |

### Conclusion

Hybrid retrieval using Reciprocal Rank Fusion achieved the strongest
overall performance across the evaluation queries.

RRF benefited from combining lexical and semantic retrieval signals.
This was particularly visible for queries such as "floating point
exception", where semantic retrieval handled vocabulary variation
better than BM25.

These results establish RRF as the baseline retrieval strategy for
the next stage of IncidentIQ.

The relevance judgments used in this experiment were constructed
using corpus-based heuristic rules, so the results should be
interpreted as an internal baseline rather than a benchmark against
human-annotated ground truth.